# Compte rendu de TP – Spark 

In [44]:
import pandas as pd 
from datetime import time
from pyspark.sql import SparkSession, functions as F

## 1. Optimisation Spark et Spark UI

In [45]:
# Session spark et chargement des datasets 
spark = SparkSession.builder.appName("spark-TP").master("local[*]").getOrCreate()
sales_big_df = spark.read.csv("sales_big.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
regions_df = spark.read.csv("regions.csv", header=True, inferSchema=True)

ConnectionRefusedError: [Errno 61] Connection refused

In [16]:
sales_big_df.head(5)

[Row(id=1, date=datetime.date(2024, 3, 6), produit='Imprimante', categorie='Informatique', quantite=14, prix_unitaire=150.0, region='Nord'),
 Row(id=2, date=datetime.date(2024, 10, 14), produit='Chargeur', categorie='Accessoires', quantite=15, prix_unitaire=15.0, region='Ouest'),
 Row(id=3, date=datetime.date(2024, 2, 9), produit='Souris', categorie='Accessoires', quantite=7, prix_unitaire=25.5, region='Ouest'),
 Row(id=4, date=datetime.date(2024, 11, 29), produit='Disque SSD', categorie='Informatique', quantite=2, prix_unitaire=120.0, region='Nord'),
 Row(id=5, date=datetime.date(2024, 1, 29), produit='Imprimante', categorie='Informatique', quantite=7, prix_unitaire=150.0, region='Sud')]

### Temps d'éxécution avec / sans cache

In [26]:
spark.catalog.clearCache()

#### Aggrégation sans cache

In [27]:
%%time 
sales_big_df.groupBy("region", "categorie").agg(F.sum("quantite").alias("total_vendu")).show()

+------+------------+-----------+
|region|   categorie|total_vendu|
+------+------------+-----------+
|  Nord| Accessoires|     130116|
|   Sud|Informatique|     129774|
| Ouest|Informatique|     130326|
|  Nord|Informatique|     132866|
|   Sud| Accessoires|     130839|
|   Est|Informatique|     133457|
|   Est| Accessoires|     128514|
| Ouest| Accessoires|     133995|
+------+------------+-----------+

CPU times: user 3.71 ms, sys: 4.28 ms, total: 7.99 ms
Wall time: 349 ms


#### Aggrégation avec cache

In [28]:
# Mise en cache
sales_big_cache = sales_big_df.cache()
sales_big_cache.count()

100000

In [29]:
%%time 
sales_big_cache.groupBy("region", "categorie").agg(F.sum("quantite").alias("total_vendu")).show()

+------+------------+-----------+
|region|   categorie|total_vendu|
+------+------------+-----------+
|  Nord| Accessoires|     130116|
|   Sud|Informatique|     129774|
| Ouest|Informatique|     130326|
|  Nord|Informatique|     132866|
|   Sud| Accessoires|     130839|
|   Est|Informatique|     133457|
|   Est| Accessoires|     128514|
| Ouest| Accessoires|     133995|
+------+------------+-----------+

CPU times: user 2.26 ms, sys: 3.16 ms, total: 5.43 ms
Wall time: 192 ms


#### Résultats : agrégation avec vs sans cache

| Configuration | Wall time |
|---|---|
| Sans cache | 349 ms |
| Avec cache | 192 ms |
| **Gain** | **1,8× plus rapide** |

**Sans cache**, Spark repart du fichier à chaque action : il relit `sales_big.csv` sur le disque, découpe chaque ligne et convertit les colonnes dans leur type (`int`, `double`, `date`). Ce n'est qu'ensuite qu'il peut faire l'agrégation.

**Avec cache**, ce travail n'est fait qu'une seule fois. Le premier `count()` stocke le DataFrame déjà lu et converti en mémoire. Les agrégations suivantes partent directement de cette copie et sautent toute l'étape de lecture.


### Partitions

In [ ]:
# Nombre initial de partitions (par défaut)
print("Partitions initialses:", sales_big_df.rdd.getNumPartitions())

Partitions initialses: 2


Le CSV est initialement lu en 2 partitions. On le redistribue en 8 partitions (`df8`), qu'on met en cache. On passe ensuite de 8 à 2 partitions de deux façons différentes, et on lance un `count()` pour déclencher le calcul.

In [ ]:
# Nombre de partitions de départ à 8 
df8 = sales_big_df.repartition(8).cache()
df8.count()
print("Partitions :", df8.rdd.getNumPartitions())

Partitions : 8


#### Avec repartition(2)


In [33]:
%%time
df_rep = df8.repartition(2)
df_rep.count()

CPU times: user 717 μs, sys: 2.99 ms, total: 3.71 ms
Wall time: 241 ms


100000

#### Avec coalesce(2)

In [34]:
%%time
df_coal = df8.coalesce(2)
df_coal.count()

CPU times: user 732 μs, sys: 2.07 ms, total: 2.8 ms
Wall time: 193 ms


100000

In [35]:
print("repartition(2) :", df_rep.rdd.glom().map(len).collect())
print("coalesce(2)    :", df_coal.rdd.glom().map(len).collect())


repartition(2) : [50000, 50000]
coalesce(2)    : [50001, 49999]


In [ ]:
df_rep.explain()
df_coal.explain()

#### Résultats : repartition vs coalesce

| | `repartition(2)` | `coalesce(2)` |
|---|---|---|
| Wall time | 241 ms | 193 ms |
| Lignes par partition | `[50000, 50000]` | `[50001, 49999]` |
| Opération | `Exchange RoundRobinPartitioning(2)` | `Coalesce 2` |
| Nombre de stages | 2 (`ResultQueryStage 2`) | 1 (`ResultQueryStage 1`) |


- Les deux méthodes donnent bien 2 partitions et le même nombre total de lignes (100 000).
- coalesce(2) est environ 50 ms plus rapide que repartition(2).
- repartition produit des partitions parfaitement égales. Celles de coalesce sont presque égales, car les 8 partitions de départ étaient déjà équilibrées.
- Le plan de repartition contient une étape Exchange (un shuffle), ce qui donne 2 stages. Celui de coalesce n'en contient pas et s'exécute en 1 seul stage.


### Jointure avec broadcast()

On joint une grosse table (big_sales_df) avec une petite table (regions_df) sur la colonne region, avec puis sans broadcast.

In [ ]:
# Désactiver broadcast auto
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

#### Jointure sans broadcast 

In [ ]:
%%time
join_no_broadcast = sales_big_cache.join(regions_df, on="region")
join_no_broadcast.count()

#### Jointure avec broadcast

In [ ]:
%%time
join_broadcast = sales_big_cache.join(F.broadcast(regions_df), on="region")
join_broadcast.count()

In [ ]:
join_no_broadcast.explain()
join_broadcast.explain()

In [ ]:
# Rétablir le réglage par défaut
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
